# 5. Testing the Fine-Tuned Model 🔍

Now that we've successfully fine-tuned our Audio-Language model, it's time to evaluate its performance! In this section, test the model using your own speech examples.

Recording and Preparing the Audio File
- Record an audio file: Use Voice Memos on Mac to record a speech example.
- Convert to WAV: Use  to convert the file to .wav.
- Resample the audio: Use torchaudio.transforms.Resample to resample the audio to 16K Hz, matching the model's training sampling rate.

Finally save it to an in-memory BytesIO object and include it in tour conversation template and feed to the model for transcription.


In [1]:
import io
import torchaudio
import torch
import torchaudio.transforms as T
import torchaudio.functional as F
from transformers import AutoProcessor, Qwen2VLAudioConfig, Qwen2VLConfig
from transformers import Qwen2VLForConditionalGenerationWithAudio
from qwen_vl_utils import fetch_audio

/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
audio_path = "../reading_test.wav"
TARGET_SR = 16_000

In [3]:
waveform, sample_rate = torchaudio.load(audio_path)
sample_rate


48000

In [4]:
num_samples = waveform.shape[1]
duration_sec = num_samples / sample_rate
print(f"Duration: {duration_sec:.2f} seconds")


Duration: 13.67 seconds


In [5]:
if sample_rate != TARGET_SR:
    waveform = F.resample(waveform, orig_freq=sample_rate, new_freq=TARGET_SR)

In [6]:
waveform = waveform.clamp(-1.0, 1.0).to(torch.float32)

In [7]:
buf = io.BytesIO()

In [8]:
torchaudio.save(
    buf,
    waveform,
    TARGET_SR,
    format="wav",
    encoding="PCM_S",
    bits_per_sample=16,
)
audio_bytes = buf.getvalue()

In [9]:
audio_values = waveform.squeeze(0)

In [10]:
audio_values.shape

torch.Size([218795])

In [11]:
audio_values.shape[0]/TARGET_SR

13.6746875

In [12]:
conversation = [
    {
        "content": [{
            "text": "You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.",
            "type": "text"
        }],
        "role": "system"
    },
    {
        "content": [{
            "audio": audio_bytes,
            "type": "audio"
        },
        {
            "text": "Transcribe this speech into text.",
            "type": "text",
        }],
        "role": "user"
    }
]

In [13]:
processor = AutoProcessor.from_pretrained(
    "mdmy/whisper_asr_finetuning",
    subfolder="qwen_w_audio_processor",
    trust_remote_code=True
)

In [14]:
BASE_ID = "Qwen/Qwen2-VL-7B-Instruct"
cfg = Qwen2VLConfig.from_pretrained(BASE_ID, trust_remote_code=True)
cfg.use_audio = True
audio_cfg = Qwen2VLAudioConfig()
cfg.audio_config = audio_cfg

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


In [15]:
model = Qwen2VLForConditionalGenerationWithAudio.from_pretrained(
    "mdmy/audio-capable-qwen2-vl",
    config=cfg,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    ignore_mismatched_sizes=True,
    trust_remote_code=True,
    device_map='auto',
)
model.eval()

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46
Loading checkpoint shards: 100%|██████████| 8/8 [00:12<00:00,  1.54s/it]
Some parameters are on the meta device because they were offloaded to the disk.
/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/.venv/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2068: UserWarning: for model.layers.8.self_attn.q_proj.lora_A.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-meta parameter in the checkpoint to a meta '
/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2068: UserWarning: for model.layers.8.self_attn.q_proj.lora_B.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in

ValueError: We need an `offload_dir` to dispatch this model according to this `device_map`, the following submodules need to be offloaded: model.layers.3, model.layers.4, model.layers.5, model.layers.6, model.layers.7, model.layers.8, model.layers.9, model.layers.10, model.layers.11, model.layers.12, model.layers.13, model.layers.14, model.layers.15, model.layers.16, model.layers.17, model.layers.18, model.layers.19, model.layers.20, model.layers.21, model.layers.22, model.layers.23, model.layers.24, model.layers.25, model.layers.26, model.layers.27, model.norm, model.rotary_emb, lm_head.

In [ ]:
text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)


In [ ]:
audio_np, sr = fetch_audio(audio_bytes)

In [ ]:
batch = processor(
    text=text,
    audios=audio_np,
    audio_sample_rate=sr,
    return_tensors="pt"
)

In [ ]:
model.eval()
with torch.inference_mode():
    gen_out = model.generate(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        audio_values=batch['audio_values'],  # consumed on step 0 by prepare_inputs_for_generation
        max_new_tokens=1024,  # Use the parameter
        do_sample=False,            # set True + temperature/top_p if you want sampling
        use_cache=True,
        return_dict_in_generate=True,
        
        # output_scores=False,        # flip on if you need token-level scores
    )
    generated_ids = gen_out.sequences